# Tidy Data

¿Por qué necesitamos tidy data?
* En general los datos vienen en diferentes formatos 
* **Todos** los proyectos de ciencia de datos van a requerir que limpies los datos 
* Data tidying consiste en estructurar las bases para facilitar el analisis 

![](imagenes/tidydata_1.jpeg)


> **Cómo usar este cuaderno:** las slides de la clase (`slides/3_Tidy.pdf`) son el *mapa conceptual* — cómo se ve cada problema, qué operación lo resuelve y las trampas de cada join. **Todo el código vive aquí**: este notebook es la referencia completa y donde hacemos los ejercicios en clase.

# ¿Qué hace que una base sea tidy?
![alternative text](imagenes/tidydata_2.jpeg)


# Los tidy data hacen que la ciencia de datos sea más eficiente
![alternative text](imagenes/tidydata_3.jpeg)

# Nuestro objetivo: transformar y wrangler los datos para que sea tidy 
![alternative text](imagenes/tidydata_4.jpeg)


# ¿Cómo hacemos que nuestros datos sean tidy?

Identificar que tipo de problema tenemos 
1. Datos dentro del nombre de las columnas 
2. Una observación en muchos renglones 
3. Una celda tiene dos datos 
4. Dos celdas tienen un solo dato 
5. Los datos están divididos a lo largo de varias columnas 
6. Los datos están divididos a lo largo de distintas tablas

## 1. Datos dentro del nombre de las columnas 

In [1]:
import pandas as pd 
base_url = "https://github.com/byuidatascience/data4python4ds/raw/master/data-raw/"
table4a = pd.read_csv("{}table4a/table4a.csv".format(base_url))

In [7]:
table4a

,country,1999,2000
0,Afghanistan,745,2666
1,Brazil,37737,80488
2,China,212258,213766


### ¿Qué está mal acá?
![alternative text](imagenes/tidytable1a.png)

### ¿Cómo lo arreglamos?
![alternative text](imagenes/tidy1b.png)

### Datos en las columnas $\implies$ `melt`

Vamos a crear un ejemplo desde cero antes de arreglar el que tenemos.

In [9]:
players = pd.DataFrame({'Game':['Athletic', 'Valencia'],
'Aubameyang':[1,3], 'Dembélé':[1,0], 'de Jong':[1,1], 
'Depay': [1,0]})
players

,Game,Aubameyang,Dembélé,de Jong,Depay
0,Athletic,1,1,1,1
1,Valencia,3,0,1,0


In [5]:
# Necesitamos que las columnas se vuelva una sola columna 
pd.melt(players, id_vars = 'Game', var_name = 'Jugador',value_name = 'Goles')

,Game,Jugador,Goles
0,Athletic,Aubameyang,1
1,Valencia,Aubameyang,3
2,Athletic,Dembélé,1
3,Valencia,Dembélé,0
4,Athletic,de Jong,1
5,Valencia,de Jong,1
6,Athletic,Depay,1
7,Valencia,Depay,0


## 2. Datos dentro del nombre de las columnas 

In [10]:
table2 = pd.read_csv("{}table2/table2.csv".format(base_url))
table2

,country,year,type,count
0,Afghanistan,1999,cases,745
1,Afghanistan,1999,population,19987071
2,Afghanistan,2000,cases,2666
3,Afghanistan,2000,population,20595360
4,Brazil,1999,cases,37737
5,Brazil,1999,population,172006362
6,Brazil,2000,cases,80488
7,Brazil,2000,population,174504898
8,China,1999,cases,212258
9,China,1999,population,1272915272


### ¿Qué está mal acá?
![alternative text](imagenes/tidytable2a.png)

### ¿Cómo lo arreglamos?
![alternative text](imagenes/tidytable2b.png)

### Una observación en varios renglones $\implies$ `pivot_table`

In [13]:
pd.melt(players, id_vars = 'Game', 
        var_name = 'Jugador', 
        value_name = 'Goles').pivot_table(index = 'Game',
                                         columns = 'Jugador',
                                         values = 'Goles').reset_index()

Jugador,Game,Aubameyang,Dembélé,Depay,de Jong
0,Athletic,1.0,1.0,1.0,1.0
1,Valencia,3.0,0.0,0.0,1.0


### EJERCICIO EN CLASE 
* Utiliza `melt` y `pivot_table` para arreglar las tablas: table2 y table4
* ¿Qué columnas elegiste?
* ¿Encontraste algún problema qué no existia en el ejemplo que mostramos antes?

In [24]:
# pd.melt(table4a, id_vars = 'country', var_name = 'year', value_name = 'cases')
pd.pivot_table(table2, index = ['country', 'year'] , columns = 'type', values = 'count').reset_index()

type,country,year,cases,population
0,Afghanistan,1999,745.0,1.998707e+07
1,Afghanistan,2000,2666.0,2.059536e+07
2,Brazil,1999,37737.0,1.720064e+08
3,Brazil,2000,80488.0,1.745049e+08
4,China,1999,212258.0,1.272915e+09
5,China,2000,213766.0,1.280429e+09


## 3. Una celda tiene dos datos 

In [25]:
table3 = pd.read_csv("{}table3/table3.csv".format(base_url))
table3

,country,year,rate
0,Afghanistan,1999,745/19987071
1,Afghanistan,2000,2666/20595360
2,Brazil,1999,37737/172006362
3,Brazil,2000,80488/174504898
4,China,1999,212258/1272915272
5,China,2000,213766/1280428583


### ¿Qué está mal acá?
![alternative text](imagenes/table3a.png)

### ¿Cómo lo arreglamos?
![alternative text](imagenes/table3b.png)

### Una celda tiene dos datos $\implies$ `str.split  + pd.concat`

In [26]:
resultados  = pd.DataFrame({'Player': 
  ['Lewandowski', 'Dembélé', 'Fati', 'Pedri'], 
  'results':['12 4', '3 5' , '3 3', '2 0']})
resultados

,Player,results
0,Lewandowski,12 4
1,Dembélé,3 5
2,Fati,3 3
3,Pedri,2 0


In [42]:
nuevas_columnas = (resultados.results.str.split(' ', expand = True).rename(columns = {0:'goles', 1:'asistencias'}))
resultados_bis = pd.concat([resultados.drop(columns = 'results'), nuevas_columnas], axis = 1)
resultados_bis

,Player,goles,asistencias
0,Lewandowski,12,4
1,Dembélé,3,5
2,Fati,3,3
3,Pedri,2,0


## 4. Datos divididos en varias columnas 

In [43]:
table5 = pd.read_csv("{}table5/table5.csv".format(base_url))
table5

,country,century,year,rate
0,Afghanistan,19,99,745/19987071
1,Afghanistan,20,0,2666/20595360
2,Brazil,19,99,37737/172006362
3,Brazil,20,0,80488/174504898
4,China,19,99,212258/1272915272
5,China,20,0,213766/1280428583


### ¿Cómo lo arreglamos?
![alternative text](imagenes/table5.png)

In [44]:
resultados_bis

,Player,goles,asistencias
0,Lewandowski,12,4
1,Dembélé,3,5
2,Fati,3,3
3,Pedri,2,0


In [46]:
resultados_bis.assign(result = resultados_bis['goles'] + ' ' +  resultados_bis['asistencias'])
resultados_bis['result'] = resultados_bis['goles'] + ' ' +  resultados_bis['asistencias']

### EJERCICIO EN CLASE 
* Arreglar las tablas: table3 y table5
* ¿Qué columnas elegiste?
* ¿Cuántas nuevas columnas necesitaste?
* ¿Qué tokens utilizaste?

In [59]:
# nuevas_columnas = (table3.rate.str.split('/', expand = True).rename(columns = {0:'cases', 1:'population'}))
# pd.concat([table3.drop(columns = 'rate'), nuevas_columnas], axis = 1)

# table5.dtypes
# table5['century'].astype(str) + table5['year'].astype(str).str.zfill(2)
table5['year'].apply(lambda x: f'{x:02d}')
#table5['YEAR'] = 


0    99
1    00
2    99
3    00
4    99
5    00
Name: year, dtype: object

## 4. Datos en distintas tablas (joins)

### Llaves:
* **primarias:** identifica de manera única una observación en su propia tabla
* **externas/extranjeras/foreign:** identifica de manera única una observación en otra tabla

#### ¿Cómo encontramos nuestra llave primaria?
* La mayoría de las bases no necesariamente nos van a decir cual o cuales son sus llaves 
* Usaremos las bases de vuelos de la clase de pandas 

In [66]:
raw_path = '../data/'

planes = pd.read_csv(raw_path + "flights/raw/planes.csv")

planes.groupby('plane').type.agg(n = 'size').query('n>1')
#planes.plane.value_counts().value_counts()


,n
plane,


* ¿Qué hicimos acá? 
* ¿Qué necesita ser cierto para que una variable sea la llave primaria?

#### ¿Qué hago si no tengo llave primaria?
* Crear una: puedes usar el número de renglón como llave 

### Relaciones entre tablas: 
* una a una 
* muchas a una
* una a muchas
* muchas a muchas 

In [67]:
goles = pd.DataFrame({'jugador':
  ['Lewandowski', 'Dembélé', 'Fati',
  'Pedri', 'Torres'],
'goles':[12, 3, 3, 2, 2]})
asistencias = pd.DataFrame({'jugador':
  ['Dembélé', 'Lewandowski', 
  'Fati', 'Balde', 'Koundé'],
'asistencias':[5, 4, 3, 3, 2]})

In [69]:
asistencias

,jugador,asistencias
0,Dembélé,5
1,Lewandowski,4
2,Fati,3
3,Balde,3
4,Koundé,2


In [71]:
goles

,jugador,goles
0,Lewandowski,12
1,Dembélé,3
2,Fati,3
3,Pedri,2
4,Torres,2


### 1. Inner 
![alternative text](imagenes/inner_join.png)

In [72]:
# merge (Module function)
pd.merge(goles, asistencias, how = 'inner',  on = 'jugador')
# join (instance method)
goles.join(asistencias.set_index('jugador'), on = 'jugador', how = 'inner')

,jugador,goles,asistencias
0,Lewandowski,12,4
1,Dembélé,3,5
2,Fati,3,3


#### Diferencias entre `merge` y `join`?
|           | `merge`     | `join`         |
|-----------| ----------- | -------------- |
|junta en:  | columnas    | indices        |
|default:   | inner       | left join      |
|ventajas:  | sin extra   | más eficiente  |


### 2. Outer joins 

Existen 3 tipos de outer joins: 

1. Izquierda
2. Derecha 
3. Full


### 2.1 Outer join: izquierda 
![alternative text](imagenes/left_join.png)


In [74]:
# Merge
pd.merge(goles, asistencias, how = 'left', on = 'jugador')
# Join
goles.join(asistencias.set_index('jugador'), 
on = 'jugador')

,jugador,goles,asistencias
0,Lewandowski,12,4.0
1,Dembélé,3,5.0
2,Fati,3,3.0
3,Pedri,2,NaN
4,Torres,2,NaN


#### 2.2 Outer join: derecha

![alternative text](imagenes/right_join.png)

In [23]:
pd.merge(goles, asistencias, how = 'right', 
on = 'jugador')

,jugador,goles,asistencias
0,Dembélé,3.0,5
1,Lewandowski,12.0,4
2,Fati,3.0,3
3,Balde,NaN,3
4,Koundé,NaN,2


### 2.3 Outer join: full

![alternative text](imagenes/full_join.png)

In [25]:
pd.merge(goles, asistencias, how = 'outer',
on = 'jugador')

,jugador,goles,asistencias
0,Lewandowski,12.0,4.0
1,Dembélé,3.0,5.0
2,Fati,3.0,3.0
3,Pedri,2.0,NaN
4,Torres,2.0,NaN
5,Balde,NaN,3.0
6,Koundé,NaN,2.0


### 3. Semi joins 
![alternative text](imagenes/semi_join.png)

In [81]:
# inner_join = pd.merge(goles, asistencias, on = 'jugador', how = 'right')
# goles[goles['jugador'].isin(inner_join['jugador'])]

RangeIndex(start=0, stop=5, step=1)

### 4. Anti joins 
![alternative text](imagenes/anti_join.png)

In [83]:
outer = pd.merge(goles, asistencias, how='outer', indicator=True)
outer
outer[outer._merge == 'left_only'].drop('_merge',axis = 1)

,jugador,goles,asistencias
5,Pedri,2.0,NaN
6,Torres,2.0,NaN


### Tipos de joins: 

* `inner`: incluye unicamente renglones que aparecen en ambas tablas 
* `left`: incluye todos los renglones de `x` y aquellos de `y` que hacen match
* `right`: incluye todos los renglones de `y` y aquellos de `x` que hacen match
* `outer`: incluter todos los renglones de `x` y `y`
* `semi`: incluye los renglones de `x` que hacen match con `y`
* `anti`: incluye los renglones de `x` que no hacen match con `y`

### Sugerencia de como usar joins: 

* **PREFERRED JOINS:** `left` y `inner`
* **NOT THAT COMMON:** `right` y `outer` (con cuidado)
* **CHECAR MESSY JOINS:** `semi` y `anti`

### EJERCICIO EN CLASE:
Utilizando las bases de datos de `flights` contesta lo siguiente:
1. ¿Qué condiciones del clima están asociadas a retrasos de vuelos que salen de Houston?
2. Los aviones más viejos son los que más se retrazan?

In [87]:
weather = pd.read_csv(raw_path + 'flights/raw/weather.csv')
flights = pd.read_csv(raw_path + 'flights/raw/flights.csv')

In [90]:
from datetime import datetime

In [121]:
weather['date'] = pd.to_datetime(weather['date'], format='%Y-%m-%d')
flights['date'] = pd.to_datetime(flights['date'].str.split(' ').str[0], format='%Y-%m-%d')
flights['hour'] = flights['hour'].astype('Int64')

In [122]:
flights.date

0        2011-01-01
1        2011-01-02
2        2011-01-03
3        2011-01-04
4        2011-01-05
            ...    
227491   2011-12-06
227492   2011-12-06
227493   2011-12-06
227494   2011-12-06
227495   2011-12-06
Name: date, Length: 227496, dtype: datetime64[ns]

In [123]:
retrasados = flights[flights.dep_delay > 60]


In [124]:
retrasados.date

16       2011-01-17
19       2011-01-20
73       2011-01-14
96       2011-01-09
98       2011-01-11
            ...    
227407   2011-12-06
227422   2011-12-06
227473   2011-12-06
227474   2011-12-06
227478   2011-12-06
Name: date, Length: 10242, dtype: datetime64[ns]

In [125]:
retrasados_weather = pd.merge(retrasados, weather, how = 'inner', on = ['date', 'hour'])

In [127]:
retrasados_weather.conditions.value_counts()

conditions
Mostly Cloudy                   2433
Scattered Clouds                2334
Overcast                        1654
Clear                           1596
Partly Cloudy                   1544
Light Rain                       277
Heavy Rain                        62
Haze                              60
Light Thunderstorms and Rain      57
Rain                              44
Thunderstorms and Rain            32
Thunderstorm                      24
Heavy Thunderstorms and Rain      19
Light Freezing Rain                8
Fog                                7
Freezing Rain                      5
Drizzle                            3
Shallow Fog                        1
Name: count, dtype: int64

In [130]:
planes_flights = pd.merge(flights, planes, how = 'left', on = 'plane')

In [131]:
planes_flights

,date,hour,minute,dep,arr,dep_delay,arr_delay,carrier,flight,dest,...,time,dist,year,mfr,model,no.eng,no.seats,speed,engine,type
0,2011-01-01,14,0.0,1400.0,1500.0,0.0,-10.0,AA,428,DFW,...,40.0,224,1991.0,MCDONNELL DOUGLAS,DC-9-82(MD-82),2.0,172.0,NaN,Turbo-fan,Fixed wing multi engine
1,2011-01-02,14,1.0,1401.0,1501.0,1.0,-9.0,AA,428,DFW,...,45.0,224,1993.0,MARZ BARRY,KITFOX IV,1.0,2.0,NaN,Reciprocating,Fixed wing single engine
2,2011-01-03,13,52.0,1352.0,1502.0,-8.0,-8.0,AA,428,DFW,...,48.0,224,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2011-01-04,14,3.0,1403.0,1513.0,3.0,3.0,AA,428,DFW,...,39.0,224,1974.0,RAVEN,S55A,NaN,1.0,60.0,NaN,Balloon
4,2011-01-05,14,5.0,1405.0,1507.0,5.0,-3.0,AA,428,DFW,...,44.0,224,1989.0,MCDONNELL DOUGLAS,DC-9-82(MD-82),2.0,172.0,NaN,Turbo-fan,Fixed wing multi engine
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227491,2011-12-06,18,18.0,1818.0,2111.0,8.0,-9.0,WN,1191,TPA,...,97.0,781,2007.0,BOEING,737-7H4,2.0,140.0,NaN,Turbo-fan,Fixed wing multi engine
227492,2011-12-06,20,47.0,2047.0,2334.0,7.0,4.0,WN,1674,TPA,...,94.0,781,1993.0,BOEING,737-3H4,2.0,149.0,NaN,Turbo-fan,Fixed wing multi engine
227493,2011-12-06,9,12.0,912.0,1031.0,-3.0,-4.0,WN,127,TUL,...,61.0,453,2000.0,BOEING,737-7H4,2.0,140.0,NaN,Turbo-fan,Fixed wing multi engine
227494,2011-12-06,6,56.0,656.0,812.0,-4.0,-13.0,WN,621,TUL,...,64.0,453,1999.0,BOEING,737-7H4,2.0,140.0,NaN,Turbo-fan,Fixed wing multi engine


In [137]:
planes_flights[planes_flights.arr_delay > 120].year.value_counts().sort_values(ascending = False)


year
2002.0    284
2001.0    278
1998.0    233
1999.0    229
2004.0    217
2003.0    205
1997.0    195
2005.0    194
2000.0    179
2008.0    106
2006.0     95
2009.0     77
1995.0     75
1996.0     74
1994.0     62
2007.0     43
2010.0     42
1988.0     37
1990.0     37
1991.0     34
1993.0     32
1992.0     30
1985.0     28
1989.0     25
1986.0     25
1987.0     20
2011.0     12
1976.0      8
1979.0      4
1977.0      4
1946.0      4
1980.0      3
1965.0      3
1978.0      2
2013.0      1
1983.0      1
Name: count, dtype: int64

# Missing Values 

Cuando volvemos nuestra base tidy pueden aparecer missing values que antes no estabamos viendo y estos pueden ser de dos tipos:

* **Explicitos:** vemos los `NA`
* **Implicitos:** no estan presentes en la base

Usaremos la siguiente base para ilustrar esto: 

In [3]:
import numpy as np
data_stocks = {'year':[2015,2015,2015,2015,2016,2016,2016],
              'qtr': [1,2,3,4,2,3,4],
              'retorno':[1.88,0.59,0.35,np.nan,0.92,0.17,2.66]}
stocks = pd.DataFrame(data_stocks)
stocks

,year,qtr,retorno
0,2015,1,1.88
1,2015,2,0.59
2,2015,3,0.35
3,2015,4,NaN
4,2016,2,0.92
5,2016,3,0.17
6,2016,4,2.66


En esta base hay dos missing values: 
   * El return del 4 cuatrimestre del 2015 falta explicitamente, hay un `NaN` ahí
   * El return del primer cuatrimestre del 2016 falta implicitamente, no esta ahí

Depende de como representemos esta base podemos hacer los valores faltantes implicitos, explicitos. 

In [4]:
stocks.pivot_table(index = 'qtr',
                   columns = 'year',
                  values = 'retorno')

year,2015,2016
qtr,,
1,1.88,NaN
2,0.59,0.92
3,0.35,0.17
4,NaN,2.66


Otra manera de encontrar los missing implicitos, es completar la base de datos y sacar la diferencia entre la base completa y la base original. 
1. Completa la base (usando el identificador único de la base)
2. Calcula la diferencia 

In [6]:
## 1. Completar la base (acá el identificar único es el año y el cuatrimestre)
all_years = stocks['year'].unique()
all_qtrs = stocks['qtr'].unique()
complete_df = pd.MultiIndex.from_product([all_years, all_qtrs], names = ['year', 'qtr']).to_frame(index = False)
implicit_stocks = pd.merge(complete_df, stocks, on = ['year', 'qtr'], how = 'left', indicator = True)
implicit_stocks

# ## 2. Calcular la diferencia de renglones 
row_difference = len(implicit_stocks) - len(stocks)
print(row_difference)

,year,qtr,retorno,_merge
0,2015,1,1.88,both
1,2015,2,0.59,both
2,2015,3,0.35,both
3,2015,4,NaN,both
4,2016,1,NaN,left_only
5,2016,2,0.92,both
6,2016,3,0.17,both
7,2016,4,2.66,both


In [27]:
implicit_stocks

,year,qtr,retorno,_merge
0,2015,1,1.88,both
1,2015,2,0.59,both
2,2015,3,0.35,both
3,2015,4,NaN,both
4,2016,1,NaN,left_only
5,2016,2,0.92,both
6,2016,3,0.17,both
7,2016,4,2.66,both
